In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import os
import shutil
import subprocess
import time
import Metashape
import requests

In [0]:
run_date = dt.datetime.now().strftime("%Y-%m-%d")

# This is the Batch File from Jorge. (click to view)
<?xml version="1.0" encoding="UTF-8"?>
<batchjobs version="2.1.1">
  <job name="AlignPhotos" enabled="false" target="all">
    <keypoint_limit>60000</keypoint_limit>
    <keypoint_limit_per_mpx>4000</keypoint_limit_per_mpx>
    <mask_tiepoints>false</mask_tiepoints>
    <tiepoint_limit>0</tiepoint_limit>
  </job>
  <job name="OptimizeCameras" enabled="false" target="all">
    <fit_b1>true</fit_b1>
    <fit_b2>true</fit_b2>
    <fit_k4>true</fit_k4>
  </job>
  <job name="BuildPointCloud" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="DetectMarkers" enabled="false" target="all">
    <tolerance>80</tolerance>
  </job>
  <job name="LocateReflectancePanels" enabled="false" target="all"/>
  <job name="CalibrateReflectance" enabled="false" target="all">
    <use_sun_sensor>true</use_sun_sensor>
  </job>
  <job name="BuildDem" enabled="false" target="all">
    <downscale>2</downscale>
    <filter_mode>3</filter_mode>
    <reuse_depth>true</reuse_depth>
  </job>
  <job name="BuildOrthomosaic" enabled="false" target="all"/>
</batchjobs>

In [0]:

# # Step 1: Start Metashape in offscreen mode (non-blocking)
# metashape_process = subprocess.Popen(
#     ["/opt/agisoft/metashape-pro/metashape.sh", "-platform", "offscreen"],
#     stdout=subprocess.DEVNULL,
#     stderr=subprocess.DEVNULL
# )

# time.sleep(15)  # Wait until metsashape is ready

print(Metashape.app.version)

 Agisoft License Key: ([REDACTED])


In [0]:
import Metashape
import os
import shutil
import re

# ----------------------------------------------------------------------
# Read the input parameter passed by the orchestrator/trigger.
# 'flight_metadata_paths' is a comma-separated string of flight paths.
# ----------------------------------------------------------------------
dbutils.widgets.text("flight_metadata_paths", "")
raw_paths = dbutils.widgets.get("flight_metadata_paths")

# If no path was received, stop the notebook gracefully (shuts down the node).
if not raw_paths:
    dbutils.notebook.exit("Error: No flight path was received from the trigger. Shutting down node.")


# Split the comma-separated string into a list of individual flight paths.
flights_list = raw_paths.split(',')


# Databricks fix: convert 'dbfs:/' prefixes into '/dbfs/' so the standard
# 'os'/'shutil' libraries can read the paths correctly.
flights_list = [p.replace("dbfs:/", "/dbfs/") for p in flights_list]

print(f" Received the order to process a batch of {len(flights_list)} flights.")

# ----------------------------------------------------------------------
# Metashape hardware configuration:
# use the GPU (mask = 1) and disable the CPU for processing.
# ----------------------------------------------------------------------
Metashape.app.gpu_mask = 1
Metashape.app.cpu_enable = False


# ======================================================================
# MAIN LOOP: process each flight one by one.
# ======================================================================
for json_path in flights_list:
    print("\n" + "="*70)
    print(f" STARTING MISSION: {json_path}")
    print("="*70)

    # Base project folder: the directory that contains the metadata JSON.
    PROJECT_DIR = os.path.dirname(json_path)

    # ------------------------------------------------------------------
    # Detect the raw image directory and the sensor type.
    # The raw data lives under 'raw_data/' in either a 'multi-spec'
    # (multispectral) or an 'rgb' subfolder.
    # ------------------------------------------------------------------
    base_raw_dir = os.path.join(PROJECT_DIR, "raw_data")
    sensor_type = ""
    if os.path.exists(os.path.join(base_raw_dir, "multi-spec")):
        RAW_IMG_DIR = os.path.join(base_raw_dir, "multi-spec")
        sensor_type = "MS"
        print("  Image directory detected: multi-spec")
    elif os.path.exists(os.path.join(base_raw_dir, "rgb")):
        RAW_IMG_DIR = os.path.join(base_raw_dir, "rgb")
        sensor_type = "RGB"
        print("  Image directory detected: rgb")
    else:
        # Neither subfolder exists: skip this flight and move on.
        print(f"  Error: Neither 'rgb' nor 'multi-spec' subfolder was found in {base_raw_dir}")
        continue  # Jump to the next flight in the list

    # ------------------------------------------------------------------
    # Build the output directory:
    #   PROJECT_DIR / orthomosaic / agisoft_YEAR_MONTH_DAY / SENSOR_TYPE
    # The date (YYYY-MM-DD) is extracted from the project path on each
    # iteration, since it changes for every flight.
    # ------------------------------------------------------------------
    date_match = re.search(r'(\d{4})-(\d{2})-(\d{2})', PROJECT_DIR)
    if date_match:
        year, month, day = date_match.groups()
        date_folder = f"agisoft_{year}_{month}_{day}"
    else:
        # Fallback name if no date is found in the path.
        print(f"  Warning: No date (YYYY-MM-DD) found in {PROJECT_DIR}. Using 'agisoft_no_date'.")
        date_folder = "agisoft_no_date"

    OUTPUTS_DIR = os.path.join(PROJECT_DIR, "orthomosaic", date_folder, sensor_type)
    # Create the output folder tree if it does not exist yet
    # (required so the final shutil.copy2 does not fail).
    os.makedirs(OUTPUTS_DIR, exist_ok=True)
    print(f"  Output directory: {OUTPUTS_DIR}")

    # ------------------------------------------------------------------
    # Local (cluster SSD) scratch paths. Working on local disk instead of
    # the Volume avoids I/O errors and speeds up processing.
    # ------------------------------------------------------------------
    flight_name = os.path.basename(PROJECT_DIR)
    local_tmp = f"/tmp/metashape_{flight_name}"
    input_images_local = f"{local_tmp}/local_images"
    outputs_local = f"{local_tmp}/outputs"

    # Processing flags.
    GCP = False                    # Whether Ground Control Points/markers are used.
    has_reflectance_panels = True  # Whether reflectance calibration panels are present.

    # Clean any leftover scratch directory from a previous run.
    if os.path.exists(local_tmp):
        shutil.rmtree(local_tmp)

    # Create fresh local input/output folders.
    os.makedirs(input_images_local, exist_ok=True)
    os.makedirs(outputs_local, exist_ok=True)

    # ------------------------------------------------------------------
    # Copy the raw images from the Volume to the local SSD.
    # ------------------------------------------------------------------
    if os.path.exists(RAW_IMG_DIR):
        pictures = [f for f in os.listdir(RAW_IMG_DIR) if f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
        print(f" Copying {len(pictures)} images into local SSD...")
        for img in pictures:
            shutil.copy2(os.path.join(RAW_IMG_DIR, img), os.path.join(input_images_local, img))
    else:
        # Safety check: skip the flight if the raw data folder is missing.
        print(f" Error: There is no raw_data folder in {PROJECT_DIR}. Skipping flight.")
        continue

    # Build the list of local image paths that Metashape will ingest.
    photos = [os.path.join(input_images_local, f) for f in os.listdir(input_images_local) if f.lower().endswith(('.tif', '.jpg', '.jpeg'))]
    print(f" Starting Metashape with {len(photos)} images...")

    try:
        # --------------------------------------------------------------
        # Activate the Agisoft license using the secret stored in the
        # Databricks secret scope.
        # --------------------------------------------------------------
        agisoft_license_key = dbutils.secrets.get(scope="agisoft_creds", key="agisoft_license_key")
        Metashape.license.activate(agisoft_license_key)

        # Create a new Metashape project and add a processing chunk.
        doc = Metashape.Document()
        doc.save(local_tmp + '/project.psx')
        chunk = doc.addChunk()

        # --------------------------------------------------------------
        # Add markers (Ground Control Points) only if GCP mode is enabled
        # and a marker GeoDataFrame is available in the environment.
        # --------------------------------------------------------------
        print('Adding markers if there are any')
        if GCP:
            if 'marker_gdf' in locals() and len(marker_gdf) > 0:
                for idx, row in marker_gdf.iterrows():
                    marker = chunk.addMarker()
                    marker.label = row['label']
                    marker.reference.location = Metashape.Vector([row.x, row.y, row.h])
                    marker.reference.enabled = True

        # Load the images into the chunk.
        print('Adding the photos')
        chunk.addPhotos(photos)
        doc.save()

        # --------------------------------------------------------------
        # Match photos: detect and match key/tie points across images.
        # --------------------------------------------------------------
        print('Matching the photos')
        chunk.matchPhotos(keypoint_limit=60000,
                          tiepoint_limit=0,
                          keypoint_limit_per_mpx=4000,
                          generic_preselection=True,
                          reference_preselection=True)
        doc.save()

        # Align cameras: estimate camera positions and build the sparse cloud.
        print('Aligning the cameras')
        chunk.alignCameras()
        doc.save()
        chunk.updateTransform()

        # Optimize the camera calibration parameters (bundle adjustment).
        print('Optimizing the cameras')
        chunk.optimizeCameras(fit_b1=True, fit_b2=True, fit_k4=True)
        doc.save()

        # Build depth maps (used later to generate the dense point cloud).
        print('Building the depth maps')
        chunk.buildDepthMaps(downscale=2, filter_mode=Metashape.NoFiltering, max_neighbors=16)
        doc.save()

        # --------------------------------------------------------------
        # Detect markers and import their reference coordinates,
        # only when GCP mode is enabled.
        # --------------------------------------------------------------
        print('Detecting markers and importing the reference points')
        if GCP:
            chunk.detectMarkers(target_type=Metashape.TargetType.CircularTarget, tolerance=80, filter_mask=False, maximum_residual=15)
            chunk.importReference()
        else:
            print('No markers')

        # --------------------------------------------------------------
        # Reflectance calibration (multispectral workflow):
        # locate the calibration panels, then calibrate reflectance.
        # --------------------------------------------------------------
        print('Locating the reflectance panels')
        if has_reflectance_panels:
            chunk.locateReflectancePanels()
            doc.save()

        print('Calibrating the reflectance panels')
        chunk.calibrateReflectance()
        doc.save()

        # Build the dense point cloud.
        print('Building point cloud')
        chunk.buildPointCloud()
        doc.save()

        # Build the Digital Elevation Model (DEM) from the point cloud.
        print('Building DEM')
        chunk.buildDem(source_data=Metashape.PointCloudData)
        doc.save()

        # Build the orthomosaic using the DEM as the surface.
        print('Building orthomosaic')
        chunk.buildOrthomosaic(surface_data=Metashape.ElevationData)
        doc.save()

        # --------------------------------------------------------------
        # Export all results to the LOCAL temporary output folder first.
        # --------------------------------------------------------------
        print('Exporting results to local temporary folder...')
        chunk.exportReport(outputs_local + '/report.pdf')

        # Export the 3D model only if it was generated.
        if chunk.model:
            chunk.exportModel(outputs_local + '/model.obj')

        # Export the DEM only if elevation data exists.
        if chunk.elevation:
            chunk.exportRaster(outputs_local + '/DEM.tif', source_data=Metashape.ElevationData)

        # Export the orthomosaic, naming the file according to the sensor type.
        if chunk.orthomosaic:
            if sensor_type == "MS":
                chunk.exportRaster(outputs_local + '/MS.tif', source_data=Metashape.OrthomosaicData)
                print("Multispectral orthomosaic exported as MS.tif")
            else:
                chunk.exportRaster(outputs_local + '/RGB.tif', source_data=Metashape.OrthomosaicData)
                print("Standard orthomosaic exported as RGB.tif")

        print(f'Processing finished, results saved to {outputs_local}.')

        # --------------------------------------------------------------
        # Upload the generated files from local disk to the flight's
        # output folder on the Volume.
        # --------------------------------------------------------------
        print(f'Uploading final results to the flight folder: {OUTPUTS_DIR}')
        generated_files = os.listdir(outputs_local)
        for file in generated_files:
            source = os.path.join(outputs_local, file)
            destination = os.path.join(OUTPUTS_DIR, file)
            shutil.copy2(source, destination)

        print(f' Mission {flight_name} completed and files successfully uploaded.')

    except Exception as e:
        # Catch any error during processing so the loop can continue
        # with the remaining flights instead of crashing entirely.
        print(f" An error occurred while processing {flight_name}: {e}")

    finally:
        # Always release the license and clean up local scratch files,
        # whether the flight succeeded or failed.
        Metashape.license.deactivate()

        if os.path.exists(local_tmp):
            shutil.rmtree(local_tmp)
            print(f" Cleaning the local SSD environment ({local_tmp}) completed.")

# End of the main loop: every pending mission has been handled.
print("\n The entire batch of pending missions has been processed.")

In [0]:

# ── Job ID to trigger ────────────────────────────────────────────────────
# This is the Databricks Job ID of "plot_clipping" (Job 3). Once this
# notebook's own work is done, it will kick off that job so the heavier
# processing pipeline can run next.
JOB_3_ID = 162763862056473  #change with the plot_clipping job ID

# ── Get the current workspace context (host + auth token) ───────────────
# This lets the notebook call the Databricks REST API on its own, using
# the same workspace/credentials it's currently running in — no need to
# hardcode a host URL or token.
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

# ── Build the API request to trigger the job ─────────────────────────────
# "run-now" starts an existing job immediately (equivalent to clicking
# "Run now" in the Databricks UI), using the job_id defined above.
url = f"{host}/api/2.1/jobs/run-now"
headers = {"Authorization": f"Bearer {token}"}
data = {"job_id": JOB_3_ID}

# ── Send the request and report the outcome ──────────────────────────────
response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    # A 200 response means Databricks accepted the request and started the run.
    print(" Trigger successful! Heavy processing Job has been started.")
else:
    # Anything else means the trigger failed — print the API's error message
    # so it's clear what went wrong (e.g. wrong job ID, permissions issue).
    print(f" Error triggering Job: {response.text}")